In [0]:
df = spark.read.table("group_project.bronze.dim_county")

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
dbutils.library.restartPython()


In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable
from common_utils.logging import get_logger
from common_utils.generic_functions import rename_columns , casting , drop_columns
import json

In [0]:
dbutils.widgets.text("path", "")
config_path =dbutils.widgets.get("path") 

In [0]:

import json

from pyspark.sql import functions as F

from common_utils.logging import get_logger
from common_utils.generic_functions import (
    rename_columns,
    casting,
    drop_columns
)

# =========================================
# 0. Logger
# =========================================

logger = get_logger("b2s_DimCounty")
logger.info("Session started")

# =========================================
# 1. CONFIG PATH
# =========================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

logger.info("Setting configurations")

# =========================================
# 2. Read Config
# =========================================

with open(config_path, "r") as file:
    config = json.load(file)

# =========================================
# 3. CONFIG Sections
# =========================================

source_config = config["source"]
target_config = config["target"]

source_name = source_config["source_name"]
source_table = source_config["source_table"]
target_table = target_config["target_table"]

logger.info("Configuration completed")

# =========================================
# 4. County Transformations
# =========================================

def dim_county_transformations(df, config):

    # -------------------------------------
    # Trim columns
    # -------------------------------------

    df = df.select(
        F.col("county_id"),
        F.trim(F.col("county_name")).alias("county_name"),
        F.trim(F.col("region")).alias("region"),
        F.trim(F.col("country")).alias("country"),
        F.trim(F.col("population_band")).alias("population_band"),
        F.trim(F.col("urban_rural")).alias("urban_rural"),
        F.col("source_file"),
        F.col("current_timestamp")
    )

    # -------------------------------------
    # Rename columns
    # -------------------------------------

    df = rename_columns(
        df,
        config.get("rename_column", {})
    )

    # -------------------------------------
    # Cast columns
    # -------------------------------------

    df = casting(
        df,
        config.get("cast_columns", {})
    )

    # -------------------------------------
    # Drop columns
    # -------------------------------------

    df = drop_columns(
        df,
        config.get("drop_columns", [])
    )

    # -------------------------------------
    # Select final columns
    # -------------------------------------

    logger.info(
        "[%s] Selecting final Silver columns",
        source_name
    )

    df = df.select(*config["select_columns"])

    return df


# =========================================
# 5. Reading Bronze Table
# =========================================

logger.info("Reading from source table [%s]", source_table)

df = spark.read.table(source_table)


# =========================================
# 6. Run Transformations
# =========================================

cleaned_df = dim_county_transformations(
    df,
    config
)

logger.info(
    "[%s] County transformation completed",
    source_name
)


# =========================================
# 7. Data Quality Checks
# =========================================

logger.info("[%s] Running data quality checks", source_name)

# Check null primary keys and county names

invalid_records = cleaned_df.filter(
    F.col("county_id").isNull()
    | F.col("county_name").isNull()
    | (F.trim(F.col("county_name")) == "")
)

# Check duplicate county IDs

duplicates = (
    cleaned_df
    .groupBy("county_id")
    .count()
    .filter(F.col("count") > 1)
)

# Validate records

if invalid_records.limit(1).count() > 0:
    raise ValueError("Null or invalid county records found")

if duplicates.limit(1).count() > 0:
    raise ValueError("Duplicate county IDs found")

logger.info("County data quality checks passed")


# =========================================
# 8. Writing into Silver Layer
# =========================================

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS group_project.silver"
)

(
    cleaned_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

logger.info(
    "[%s] Successfully loaded into Silver: %s",
    source_name,
    target_table
)
# I didnt apply the SCD type 2 in this table as it is statc table